In [ ]:
import kagglehub
shrutimechlearn_churn_modelling_path = kagglehub.dataset_download('shrutimechlearn/churn-modelling')

print('Data source import complete.')

Using Colab cache for faster access to the 'churn-modelling' dataset.
Data source import complete.


In [ ]:
import numpy as np
import pandas as pd


import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

import kagglehub

path = kagglehub.dataset_download('shrutimechlearn/churn-modelling')
print("Path to dataset files:", path)

/kaggle/input/churn-modelling/Churn_Modelling.csv
Using Colab cache for faster access to the 'churn-modelling' dataset.
Path to dataset files: /kaggle/input/churn-modelling


In [ ]:
data = pd.read_csv("/kaggle/input/churn-modelling/Churn_Modelling.csv")

In [ ]:
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


I can safely remove the columns RowNumber, CustomerID and Surname as it will not have an effect on churn

In [ ]:
data = data.drop(columns=['RowNumber', 'CustomerId', 'Surname'])

In [ ]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
data.isnull().sum()

,0
CreditScore,0
Geography,0
Gender,0
Age,0
Tenure,0
Balance,0
NumOfProducts,0
HasCrCard,0
IsActiveMember,0
EstimatedSalary,0


In [ ]:
data.groupby('Geography')['Exited'].mean()

,Exited
Geography,
France,0.161548
Germany,0.324432
Spain,0.166734


First I will split the data into train, validation and test sets. Then I will apply One hot encoding to the Country and Gender column using sklearn's onehotencoding

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Separate features and target
X = data.drop(columns=['Exited'])
y = data['Exited']

# 70% train, 15% validation and 15% test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42)

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['Geography', 'Gender'])
    ],
    remainder='passthrough'  # Numerical columns are left exactly as they are
)


X_train_tree = preprocessor.fit_transform(X_train)
X_val_tree = preprocessor.transform(X_val)
X_test_tree = preprocessor.transform(X_test)

In [ ]:
X_train_tree

array([[0.0000000e+00, 0.0000000e+00, 1.0000000e+00, ..., 1.0000000e+00,
        0.0000000e+00, 9.2826350e+04],
       [0.0000000e+00, 1.0000000e+00, 0.0000000e+00, ..., 1.0000000e+00,
        0.0000000e+00, 6.2027900e+04],
       [0.0000000e+00, 1.0000000e+00, 0.0000000e+00, ..., 0.0000000e+00,
        0.0000000e+00, 1.1281379e+05],
       ...,
       [0.0000000e+00, 0.0000000e+00, 1.0000000e+00, ..., 1.0000000e+00,
        1.0000000e+00, 8.0591180e+04],
       [0.0000000e+00, 0.0000000e+00, 1.0000000e+00, ..., 1.0000000e+00,
        1.0000000e+00, 1.7743159e+05],
       [1.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 1.0000000e+00,
        0.0000000e+00, 4.2720000e+03]])

In [ ]:
column_names = preprocessor.get_feature_names_out()

X_train_tree_df = pd.DataFrame(X_train_tree, columns=column_names)

X_train_tree_df.head()

,cat__Geography_France,cat__Geography_Germany,cat__Geography_Spain,cat__Gender_Female,cat__Gender_Male,remainder__CreditScore,remainder__Age,remainder__Tenure,remainder__Balance,remainder__NumOfProducts,remainder__HasCrCard,remainder__IsActiveMember,remainder__EstimatedSalary
0,0.0,0.0,1.0,1.0,0.0,680.0,37.0,6.0,124140.57,2.0,1.0,0.0,92826.35
1,0.0,1.0,0.0,0.0,1.0,725.0,40.0,8.0,104149.66,1.0,1.0,0.0,62027.90
2,0.0,1.0,0.0,0.0,1.0,529.0,29.0,4.0,135759.40,1.0,0.0,0.0,112813.79
3,0.0,0.0,1.0,1.0,0.0,637.0,54.0,5.0,0.00,1.0,0.0,1.0,150836.98
4,1.0,0.0,0.0,0.0,1.0,626.0,33.0,8.0,0.00,2.0,1.0,0.0,138504.28


In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report


In [ ]:
model_xgb = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight = 3.6, #the data is unbalanced, adding this penalty so the loss funtion focuses on churn too (there are 3.6 times as many customers that stay than leave)
    random_state=42,
    eval_metric='logloss'
)

In [ ]:
model_xgb.fit(X_train_tree, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
y_val_pred = model_xgb.predict(X_val_tree)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("\nClassification Report:\n", classification_report(y_val, y_val_pred))

Validation Accuracy: 0.8141239173884077

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.84      0.88      1178
           1       0.55      0.73      0.63       323

    accuracy                           0.81      1501
   macro avg       0.73      0.78      0.75      1501
weighted avg       0.84      0.81      0.82      1501



In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [50, 100, 150, 200],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}


random_search = RandomizedSearchCV(
    estimator=xgb.XGBClassifier(scale_pos_weight=3.6, random_state=42, eval_metric='logloss'),
    param_distributions=param_dist,
    n_iter=15,
    scoring='f1',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

print("Searching for the best hyperparameters...")
random_search.fit(X_train_tree, y_train)

# 4. View the winning parameters
print("\nBest Hyperparameters Found:", random_search.best_params_)

Searching for the best hyperparameters...
Fitting 3 folds for each of 15 candidates, totalling 45 fits

Best Hyperparameters Found: {'subsample': 0.8, 'n_estimators': 50, 'max_depth': 7, 'learning_rate': 0.01, 'colsample_bytree': 0.7}


In [ ]:
best_model_xgb = random_search.best_estimator_

In [ ]:
y_val_pred = best_model_xgb.predict(X_val_tree)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("\nClassification Report:\n", classification_report(y_val, y_val_pred))

Validation Accuracy: 0.8387741505662891

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.88      0.90      1178
           1       0.61      0.70      0.65       323

    accuracy                           0.84      1501
   macro avg       0.76      0.79      0.77      1501
weighted avg       0.85      0.84      0.84      1501



RANDOM FOREST

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
model_rf = RandomForestClassifier(
    n_estimators=100,       # Number of individual decision trees in the forest
    max_depth=5,            # Maximum depth allowed for each tree
    class_weight='balanced',# Automatically handles data imbalance based on class frequencies
    random_state=42,        # Ensures identical tree construction every run
    n_jobs=-1               # Parallel processing: uses all available CPU cores
)

In [ ]:
X_train_tree.shape, X_val_tree.shape

((6999, 13), (1501, 13))

In [ ]:
model_rf.fit(X_train_tree, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=5, n_jobs=-1,
                       random_state=42)

In [140]:
y_val_pred_rand_forest = model_rf.predict(X_val_tree)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred_rand_forest))
print("\nClassification Report:\n", classification_report(y_val, y_val_pred_rand_forest))

Validation Accuracy: 0.8021319120586275

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.81      0.86      1178
           1       0.53      0.79      0.63       323

    accuracy                           0.80      1501
   macro avg       0.73      0.80      0.75      1501
weighted avg       0.85      0.80      0.81      1501



In [ ]:
param_dist_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 8, 11, 14],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2', None]
}


random_search_rf = RandomizedSearchCV(
    estimator=RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    param_distributions=param_dist_rf,
    n_iter=10,
    scoring='f1',
    cv=3,
    random_state=42,
    n_jobs=-1
)

random_search_rf.fit(X_train_tree, y_train)

RandomizedSearchCV(cv=3,
                   estimator=RandomForestClassifier(class_weight='balanced',
                                                    n_jobs=-1,
                                                    random_state=42),
                   n_jobs=-1,
                   param_distributions={'max_depth': [5, 8, 11, 14],
                                        'max_features': ['sqrt', 'log2', None],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [100, 200, 300]},
                   random_state=42, scoring='f1')

In [ ]:
print("\nBest Hyperparameters Found:", random_search_rf.best_params_)


Best Hyperparameters Found: {'n_estimators': 300, 'min_samples_split': 10, 'max_features': 'sqrt', 'max_depth': 11}


In [ ]:
best_model_rf = random_search_rf.best_estimator_

In [ ]:
X_train_tree.shape, X_val_tree.shape

((6999, 13), (1501, 13))

In [ ]:
y_val_pred = best_model_rf.predict(X_val_tree)

print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("\nClassification Report:\n", classification_report(y_val, y_val_pred))

Validation Accuracy: 0.8500999333777481

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.90      0.90      1178
           1       0.65      0.66      0.65       323

    accuracy                           0.85      1501
   macro avg       0.78      0.78      0.78      1501
weighted avg       0.85      0.85      0.85      1501



In [141]:
y_test_preds = best_model_rf.predict(X_test_tree)

In [142]:
print("Validation Accuracy:", accuracy_score(y_test, y_test_preds))
print("\nClassification Report:\n", classification_report(y_test, y_test_preds))

Validation Accuracy: 0.8486666666666667

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.89      0.90      1207
           1       0.60      0.67      0.63       293

    accuracy                           0.85      1500
   macro avg       0.76      0.78      0.77      1500
weighted avg       0.86      0.85      0.85      1500

